# 3. GRPO Tutor A (stage 4)

**Settings (right sidebar), before running:**
1. Accelerator: **GPU T4 x2** (not P100: this pipeline refuses it).
2. Internet: **on** (needs a phone-verified account).
3. Add Input:
   - **tutorbias-data**
   - (when resuming) this notebook's own previous Output

Run with **Save Version > Save & Run All (Commit)** so it survives closing the tab. Expected GPU time: **~15 min smoke + ~3-4 h training**.
Every step is resumable: if the run stops early, attach this notebook's own output as an extra Input
and run it again; `restore` picks up the partial files and each step continues where it stopped.

Policy: Qwen2.5-1.5B-Instruct + LoRA in fp16 on GPU 0. Reward: the 7B judge in 4-bit on GPU 1. The smoke test prints seconds per step; if it runs out of memory or is slower than ~60 s/step, set `POLICY=Qwen/Qwen2.5-0.5B-Instruct` in the training cell. Checkpoints every 25 steps; training stops itself at 11 h.

In [ ]:
# Secrets (sidebar: Add-ons > Secrets). HF_TOKEN is optional (the models are public, but a token avoids
# download rate limits); WANDB_API_KEY is optional (without it, metrics are only written to files).
import os

try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for k in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[k] = s.get_secret(k)
            print(k, "loaded")
        except Exception:
            print(k, "not set (fine)")
except ImportError:
    print("not on Kaggle")
# Big files go to /tmp (the 20 GB /kaggle/working is saved as the notebook's output; keep it small).
os.environ.update(HF_HOME="/tmp/hf", VENV_ROOT="/tmp", UV_CACHE_DIR="/tmp/uv", PYTHONUNBUFFERED="1")

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d xai_proj2 ] || git clone --depth 1 https://github.com/anvitasuresh/xai_proj2
cd xai_proj2 && git pull --ff-only && git log --oneline -1
nvidia-smi --query-gpu=name,memory.total --format=csv
# Copy data/ and results/ from every attached Input (your data dataset, earlier notebooks' outputs).
PYTHONPATH=src python3 -m tutorbias.kaggle restore --repo .
PYTHONPATH=src python3 -m tutorbias.kaggle check
bash scripts/setup_local.sh gpu > /tmp/setup.log 2>&1 || (tail -30 /tmp/setup.log; exit 1)
echo "GPU env ready"

In [ ]:
%%bash
cd /kaggle/working/xai_proj2
bash scripts/run_stage.sh grpo-data

In [ ]:
%%bash
cd /kaggle/working/xai_proj2
bash scripts/run_stage.sh grpo-smoke 2>&1 | tail -60
rm -rf results/tutor/smoke/merged   # smoke weights aren't needed

In [ ]:
%%bash
cd /kaggle/working/xai_proj2
RUN=tutor_a bash scripts/run_stage.sh grpo   # add POLICY=Qwen/Qwen2.5-0.5B-Instruct if needed

In [ ]:
%%bash
cd /kaggle/working/xai_proj2
ls results/tutor/tutor_a && cat results/tutor/tutor_a/run_info.json

In [ ]:
%%bash
# Keep the output small and self-contained: drop the venv symlinks (they point into /tmp).
cd /kaggle/working/xai_proj2 && rm -f .venv-gpu .venv-data
du -sh data results 2>/dev/null || true